# 0. GPU smoke test (about 10 to 15 min)

Synthetic data only: checks feature extraction, training (incl. resume), fp16 AMP,
inference, plots and export on Kaggle's GPU image, and measures it/s.
Settings: **Accelerator: GPU**, **Internet: On**.

In [ ]:
import os, subprocess, glob, shutil, json, sys, time
REPO_URL = "https://github.com/snehithareddy28/fastspeech2-assignment.git"
os.chdir("/kaggle/working")
if not os.path.exists("fastspeech2-assignment"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir("/kaggle/working/fastspeech2-assignment")
subprocess.run(["git", "pull"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
os.environ["TQDM_MININTERVAL"] = "60"   # keep progress bars from flooding the log
os.environ["PYTHONUNBUFFERED"] = "1"


def sh(cmd):
    """Run a bash command, stream its output live, raise if it fails."""
    print("$", cmd, flush=True)
    t0 = time.time()
    p = subprocess.Popen(["bash", "-c", "set -euo pipefail; " + cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError("command failed (exit {}): {}".format(p.returncode, cmd))
    print("[ok in {:.1f} min]".format((time.time() - t0) / 60), flush=True)

In [ ]:
sh("pip install -q -r requirements.txt")
import torch
assert torch.cuda.is_available(), "No CUDA GPU visible: set the notebook accelerator to GPU"
for i in range(torch.cuda.device_count()):
    cap = torch.cuda.get_device_capability(i)
    print(i, torch.cuda.get_device_name(i), "sm_%d%d" % cap)
    # fail fast if this torch build has no kernels for the GPU (e.g. P100 = sm_60)
    x = torch.randn(256, 256, device=f"cuda:{i}"); (x @ x).sum().item()
print("torch", torch.__version__, "cuda", torch.version.cuda, "arch list", torch.cuda.get_arch_list())

In [ ]:
sh("python scripts/smoke_test.py --steps 400 --utts 200 --amp --out /kaggle/working/smoke_out")